In [35]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os


In [36]:
# 1. UPLOAD GENERATED STORES (from route_generator.ipynb)
# Load the actual optimized stores instead of generating random ones
output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
df_stores = pd.read_csv(os.path.join(output_dir, 'stores.csv'))

print(f"Loaded {len(df_stores)} stores with pre-assigned routes.")

Loaded 50 stores with pre-assigned routes.


In [ ]:
# 2. GENERATE ASSETS (The Laundry Jugs)
# 2026 ESG Specs: Every jug has a 'birth weight' and material type
assets_data = []
num_of_jugs = 100000 #update to 5,000 and eventually 1,000,000 for pilot study
for i in range(num_of_jugs):
    assets_data.append([
        f"JUG_{i:05d}", 
        "HDPE_Plastic", 
        120, # grams of plastic
        "2025-11-01" # Launch date
    ])
df_assets = pd.DataFrame(assets_data, columns=['asset_id', 'material', 'weight_grams', 'initial_date'])

#print (df_assets) 


In [38]:
# --- ROUTE-BASED LOGISTICS ---
events = []
simulation_start_date = datetime(2026, 1, 1)

# Group stores by their Route ID for batching
routes_dict = df_stores.groupby('route_id')['store_id'].apply(list).to_dict()

for jug_id in df_assets['asset_id']:
    current_time = simulation_start_date + timedelta(days=np.random.randint(0, 60))
    rotations = np.random.randint(1, 11)
    
    for r in range(rotations):
        # 1. Pick a ROUTE instead of just a random store
        route_id = np.random.choice(list(routes_dict.keys()))
        # 2. Pick a random store WITHIN that route
        store_id = np.random.choice(routes_dict[route_id])
        
        # --- AT RETAIL & AT HOME ---
        current_time += timedelta(days=np.random.randint(2, 14)) # Shelf
        events.append([jug_id, store_id, 'purchased', current_time])
        
        current_time += timedelta(days=np.random.randint(45, 91)) # Usage
        
        # --- RETURN & HUB PROCESSING ---
        if np.random.random() < 0.90:
            current_time += timedelta(days=np.random.randint(3, 15)) # Return Lag
            events.append([jug_id, store_id, 'returned_to_store', current_time])
            
            # Jug returns when route is picked up (simulated by +3 days)
            current_time += timedelta(days=3) 
            events.append([jug_id, 'HUB_AURORA', 'washed', current_time])
        else:
            events.append([jug_id, 'UNKNOWN', 'lost_to_leakage', current_time])
            break
        
df_events = pd.DataFrame(events, columns=['asset_id', 'location_id', 'event_type', 'timestamp'])
#print(df_events.head())

In [41]:
#Upload data to Postgres Database
from sqlalchemy import create_engine, text
from io import StringIO
import os
import pandas as pd

# --- 1. THE FUNCTION ---
def postgres_upsert(df, table_name, engine):
    output = StringIO()
    df.to_csv(output, sep='\t', header=False, index=False)
    output.seek(0)
    
    raw_conn = engine.raw_connection()
    cursor = raw_conn.cursor()
    try:
        # FIX: We remove the extra f'"{col}"' and just use the column names
        columns = [col for col in df.columns]
        
        cursor.copy_from(output, table_name, sep='\t', null='', columns=columns)
        raw_conn.commit()
        print(f"Successfully uploaded {len(df)} rows to {table_name}.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Error uploading to {table_name}: {e}")
    finally:
        cursor.close()
        raw_conn.close()

# --- 2. SETUP ---
engine = create_engine('postgresql+psycopg2://postgres@localhost:5432/aurora_reuse_jug_db')
target_folder = 'data'

# --- 3. SAVE CSV BACKUPS ---
# We save the file as stores.csv to stay consistent
df_assets.to_csv(os.path.join(target_folder, 'assets.csv'), index=False)
df_events.to_csv(os.path.join(target_folder, 'logistics_events.csv'), index=False)
df_stores.to_csv(os.path.join(target_folder, 'stores.csv'), index=False)

# --- 4. REFRESH DATABASE ---
try:
    with engine.connect() as conn:
        # Changed back to 'stores'
        conn.execute(text("TRUNCATE TABLE logistics_events, assets, stores CASCADE;"))
        conn.commit()
        print("Database cleared. Starting upload...")

    # --- 5. EXECUTE UPLOADS ---
    # We use 'stores' as the table name here
    postgres_upsert(df_stores, 'stores', engine)
    postgres_upsert(df_assets, 'assets', engine)
    postgres_upsert(df_events, 'logistics_events', engine)
    
    print("All tables synchronized successfully using table name: stores")

except Exception as e:
    print(f"Main Process Error: {e}")

Database cleared. Starting upload...
Successfully uploaded 50 rows to stores.
Successfully uploaded 100000 rows to assets.
Successfully uploaded 1203042 rows to logistics_events.
All tables synchronized successfully using table name: stores
